In [60]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import load_npz
from pathlib import Path

import gc as gc
from time import perf_counter

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [53]:
RUTA_PROYECTO = Path("..")

RUTA_PRIMERA_ITERACION = (
    RUTA_PROYECTO
    / "models"
    / "primera_iteracion"
)

RUTA_NAIVE_BAYES = (
    RUTA_PRIMERA_ITERACION
    / "naive_bayes"
)

RUTA_REGRESION_LOGISTICA = (
    RUTA_PRIMERA_ITERACION
    / "regresion_logistica"
)

RUTA_SEGUNDA_ITERACION = (
    RUTA_PROYECTO
    / "models"
    / "segunda_iteracion"
)

RUTA_ANALISIS_ERRORES = (
    RUTA_SEGUNDA_ITERACION
    / "analisis_errores"
)

RUTA_ANALISIS_ERRORES.mkdir(
    parents=True,
    exist_ok=True
)

RUTA_VECTORIZACION = (
    RUTA_PROYECTO
    / "models"
    / "primera_iteracion"
    / "vectorizacion"
)

X_train_tfidf = load_npz(
    RUTA_VECTORIZACION
    / "X_train_tfidf.npz"
)

y_train = np.load(
    RUTA_VECTORIZACION
    / "y_train.npy",
    allow_pickle=True
)

print(
    "Matriz de entrenamiento:",
    X_train_tfidf.shape
)

print(
    "Etiquetas de entrenamiento:",
    len(y_train)
)

print(
    "¿Coinciden?",
    X_train_tfidf.shape[0] == len(y_train)
)
print(
    "Carpeta de análisis:",
    RUTA_ANALISIS_ERRORES.resolve()
)

Matriz de entrenamiento: (1188274, 50000)
Etiquetas de entrenamiento: 1188274
¿Coinciden? True
Carpeta de análisis: C:\Users\Paula María\Desktop\IS\5º CURSO\TFG\Trabajo-Fin-de-Grado\models\segunda_iteracion\analisis_errores


In [3]:
ruta_predicciones_nb = (
    RUTA_NAIVE_BAYES
    / "resultados_predicciones_naive_bayes.parquet"
)

ruta_predicciones_rl = (
    RUTA_REGRESION_LOGISTICA
    / "resultados_predicciones_regresion_logistica.parquet"
)

print(
    "Predicciones de Naive Bayes:",
    ruta_predicciones_nb.exists()
)

print(
    "Predicciones de Regresión Logística:",
    ruta_predicciones_rl.exists()
)

Predicciones de Naive Bayes: True
Predicciones de Regresión Logística: True


In [4]:
# Cargar predicciones

resultados_nb = pd.read_parquet(
    ruta_predicciones_nb
)

resultados_rl = pd.read_parquet(
    ruta_predicciones_rl
)

print(
    "Resultados de Naive Bayes:",
    resultados_nb.shape
)

print(
    "Resultados de Regresión Logística:",
    resultados_rl.shape
)

Resultados de Naive Bayes: (297069, 4)
Resultados de Regresión Logística: (297069, 4)


In [5]:
print(
    "Columnas de Naive Bayes:",
    resultados_nb.columns.tolist()
)

print(
    "Columnas de Regresión Logística:",
    resultados_rl.columns.tolist()
)

Columnas de Naive Bayes: ['review_text_processed', 'sentiment', 'sentiment_predicted', 'prediccion_correcta']
Columnas de Regresión Logística: ['review_text_processed', 'sentiment', 'sentiment_predicted', 'prediccion_correcta']


In [6]:
# Renombrar columnas para luego juntarlas y compararlas

resultados_nb = resultados_nb.rename(
    columns={
        "sentiment_predicted": "prediccion_nb",
        "prediccion_correcta": "correcta_nb"
    }
)

resultados_rl = resultados_rl.rename(
    columns={
        "sentiment_predicted": "prediccion_rl",
        "prediccion_correcta": "correcta_rl"
    }
)

In [9]:
resultados_nb = resultados_nb[
    [
        "review_text_processed",
        "sentiment",
        "prediccion_nb",
        "correcta_nb"
    ]
].copy()

resultados_rl = resultados_rl[
    [
       "review_text_processed",
        "sentiment",
        "prediccion_rl",
        "correcta_rl"
    ]
].copy()

In [10]:
# Comprobar que tienen el mismo orden

print(
    "¿Mismo número de filas?",
    len(resultados_nb) == len(resultados_rl)
)

print(
    "¿Los textos están en el mismo orden?",
    resultados_nb["review_text_processed"].equals(
        resultados_rl["review_text_processed"]
    )
)

print(
    "¿Las etiquetas reales coinciden?",
    resultados_nb["sentiment"].equals(
        resultados_rl["sentiment"]
    )
)


¿Mismo número de filas? True
¿Los textos están en el mismo orden? True
¿Las etiquetas reales coinciden? True


In [34]:
# Crear comparación

comparacion_predicciones = pd.DataFrame({
    "review_text_processed": resultados_nb["review_text_processed"],
    "sentiment": resultados_nb["sentiment"],
    "prediccion_nb": resultados_nb["prediccion_nb"],
    "prediccion_rl": resultados_rl["prediccion_rl"],
    "correcta_nb": resultados_nb["correcta_nb"],
    "correcta_rl": resultados_rl["correcta_rl"]
})


In [35]:
# Clasificar resultados

comparacion_predicciones["resultado_comparacion"] = np.select(
    [
        (
            comparacion_predicciones["correcta_nb"]
            & comparacion_predicciones[
                "correcta_rl"
            ]
        ),
        (
            comparacion_predicciones["correcta_nb"]
            & ~comparacion_predicciones[
                "correcta_rl"
            ]
        ),
        (
            ~comparacion_predicciones["correcta_nb"]
            & comparacion_predicciones[
                "correcta_rl"
            ]
        )
    ],
    [
        "Ambos aciertan",
        "Solo acierta Naive Bayes",
        "Solo acierta Regresión Logística"
    ],
    default="Ambos fallan"
)

In [36]:
resumen_comparacion = (
    comparacion_predicciones[
        "resultado_comparacion"
    ]
    .value_counts()
    .rename_axis("Resultado")
    .reset_index(name="Cantidad")
)

resumen_comparacion["Porcentaje"] = (
    resumen_comparacion["Cantidad"]
    / len(comparacion_predicciones)
    * 100
).round(2).astype("str")+"%"

resumen_comparacion

,Resultado,Cantidad,Porcentaje
0,Ambos aciertan,247115,83.18%
1,Ambos fallan,26946,9.07%
2,Solo acierta Regresión Logística,16964,5.71%
3,Solo acierta Naive Bayes,6044,2.03%


In [37]:
ventaja_rl = 16_964 - 6_044

print(
    "La Regresión Logística corrige un total de:",
    f"{ventaja_rl:,}", "reseñas más que Naive Bayes"
)

La Regresión Logística corrige un total de: 10,920 reseñas más que Naive Bayes


In [38]:
# Comprobación de errores
errores_compartidos = (
    comparacion_predicciones.loc[
        comparacion_predicciones[
            "resultado_comparacion"
        ].eq("Ambos fallan")
    ]
    .copy()
)

distribucion_errores_compartidos = (
    errores_compartidos["sentiment"]
    .value_counts()
    .rename_axis("Sentimiento real")
    .reset_index(name="Cantidad")
)

distribucion_errores_compartidos[
    "Porcentaje"
] = (
    distribucion_errores_compartidos[
        "Cantidad"
    ]
    / len(errores_compartidos)
    * 100
).round(2).astype(str)+"%"

distribucion_errores_compartidos

,Sentimiento real,Cantidad,Porcentaje
0,neutral,16006,59.4%
1,negativo,7229,26.83%
2,positivo,3711,13.77%


In [39]:
errores_por_clase = (
    errores_compartidos["sentiment"]
    .value_counts()
)

total_por_clase = (
    comparacion_predicciones["sentiment"]
    .value_counts()
)

tasa_error_compartido = (
    errores_por_clase
    / total_por_clase
    * 100
).round(2).astype(str)+"%"

resumen_error_clase = pd.DataFrame({
    "Errores compartidos": errores_por_clase,
    "Total en prueba": total_por_clase,
    "Tasa de error compartido (%)": tasa_error_compartido
})

resumen_error_clase

,Errores compartidos,Total en prueba,Tasa de error compartido (%)
sentiment,,,
negativo,7229,47697,15.16%
neutral,16006,23558,67.94%
positivo,3711,225814,1.64%


In [ ]:
# Con que clase se confunden principalmente
confusiones_errores = (
    errores_compartidos[
        [
            "sentiment",
            "prediccion_nb",
            "prediccion_rl"
        ]
    ]
    .value_counts()
    .to_frame("Cantidad")
    .reset_index()
)

confusiones_errores

,sentiment,prediccion_nb,prediccion_rl,Cantidad
0,neutral,positivo,positivo,10173
1,negativo,positivo,positivo,4771
2,neutral,negativo,negativo,3300
3,positivo,negativo,negativo,2178
4,neutral,positivo,negativo,1755
5,negativo,neutral,neutral,1116
6,positivo,neutral,neutral,1088
7,negativo,positivo,neutral,1014
8,neutral,negativo,positivo,778
9,negativo,neutral,positivo,328


In [45]:
errores_neutrales = errores_compartidos[
    errores_compartidos["sentiment"] == "neutral"
]

pd.crosstab(
    errores_neutrales["prediccion_nb"],
    errores_neutrales["prediccion_rl"],
    rownames=["Predicción N.B."],
    colnames=["Predicción R.L."]
)

Predicción R.L.,negativo,positivo
Predicción N.B.,,
negativo,3300,778
positivo,1755,10173


In [71]:
# Prueba parámetros TF-IDF

configuraciones_tfidf = [
    {
        "nombre": "configuracion_base",
        "max_features": 50_000,
        "min_df": 5,
        "max_df": 0.95,
        "ngram_range": (1, 2),
        "sublinear_tf": True
    },
    {
        "nombre": "solo_unigramas",
        "max_features": 50_000,
        "min_df": 5,
        "max_df": 0.95,
        "ngram_range": (1, 1),
        "sublinear_tf": True
    },
    {
        "nombre": "vocabulario_ampliado",
        "max_features": 75_000,
        "min_df": 5,
        "max_df": 0.95,
        "ngram_range": (1, 2),
        "sublinear_tf": True
    },
    {
        "nombre": "menor_frecuencia_minima",
        "max_features": 50_000,
        "min_df": 3,
        "max_df": 0.95,
        "ngram_range": (1, 2),
        "sublinear_tf": True
    }
]


In [72]:
datos_train = pd.read_parquet(
    RUTA_VECTORIZACION / "datos_train.parquet"
)

X_train = datos_train[
    "review_text_processed"
]

y_train = datos_train[
    "sentiment"
]

print("Textos:", len(X_train))
print("Etiquetas:", len(y_train))
print("Tipo de un texto:", type(X_train.iloc[0]))
print("Ejemplo:", X_train.iloc[0])

Textos: 1188274
Etiquetas: 1188274
Tipo de un texto: <class 'str'>
Ejemplo: good football game quick shipping great condition good football date love


In [75]:
(
    X_train_interno,
    X_validacion,
    y_train_interno,
    y_validacion
) = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

In [76]:
print("DIVISIÓN INTERNA")
print("----------------")

print(
    "Entrenamiento interno:",
    X_train_interno.shape
)

print(
    "Validación:",
    X_validacion.shape
)

print(
    "Etiquetas de entrenamiento interno:",
    len(y_train_interno)
)

print(
    "Etiquetas de validación:",
    len(y_validacion)
)

print(
    "¿Coincide entrenamiento interno?",
    X_train_interno.shape[0]
    == len(y_train_interno)
)

print(
    "¿Coincide validación?",
    X_validacion.shape[0]
    == len(y_validacion)
)

DIVISIÓN INTERNA
----------------
Entrenamiento interno: (950619,)
Validación: (237655,)
Etiquetas de entrenamiento interno: 950619
Etiquetas de validación: 237655
¿Coincide entrenamiento interno? True
¿Coincide validación? True


In [77]:
# Experimentar con los cmpos

resultados_ajuste = []

for numero, configuracion in enumerate(
    configuraciones_tfidf,
    start=1
):
    print(
        f"\nCONFIGURACIÓN {numero}/"
        f"{len(configuraciones_tfidf)}"
    )
    print(configuracion["nombre"])
    print("-" * 50)

    # Crear el vectorizador
    vectorizador = TfidfVectorizer(
        max_features=configuracion["max_features"],
        min_df=configuracion["min_df"],
        max_df=configuracion["max_df"],
        ngram_range=configuracion["ngram_range"],
        sublinear_tf=configuracion["sublinear_tf"],
        dtype=np.float32
    )

    # Ajustar y transformar el entrenamiento interno
    inicio_vectorizacion = perf_counter()

    X_train_interno_tfidf = (
        vectorizador.fit_transform(
            X_train_interno
        )
    )

    # Transformar validación sin volver a ajustar
    X_validacion_tfidf = (
        vectorizador.transform(
            X_validacion
        )
    )

    tiempo_vectorizacion = (
        perf_counter() - inicio_vectorizacion
    )

    # Entrenar siempre el mismo modelo
    modelo_temporal = MultinomialNB(
        alpha=1.0
    )

    inicio_entrenamiento = perf_counter()

    modelo_temporal.fit(
        X_train_interno_tfidf,
        y_train_interno
    )

    tiempo_entrenamiento = (
        perf_counter() - inicio_entrenamiento
    )

    # Predecir sobre validación
    predicciones_validacion = (
        modelo_temporal.predict(
            X_validacion_tfidf
        )
    )

    # Calcular métricas
    accuracy = accuracy_score(
        y_validacion,
        predicciones_validacion
    )

    precision_macro = precision_score(
        y_validacion,
        predicciones_validacion,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_validacion,
        predicciones_validacion,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_validacion,
        predicciones_validacion,
        average="macro",
        zero_division=0
    )

    resultados_ajuste.append({
        "Configuración": configuracion["nombre"],
        "max_features": configuracion["max_features"],
        "min_df": configuracion["min_df"],
        "max_df": configuracion["max_df"],
        "ngram_range": str(
            configuracion["ngram_range"]
        ),
        "sublinear_tf": configuracion["sublinear_tf"],
        "Vocabulario real": len(
            vectorizador.vocabulary_
        ),
        "Accuracy": accuracy,
        "Precision macro": precision_macro,
        "Recall macro": recall_macro,
        "F1 macro": f1_macro,
        "Tiempo vectorización": tiempo_vectorizacion,
        "Tiempo entrenamiento": tiempo_entrenamiento
    })

    print(
        "Vocabulario:",
        f"{len(vectorizador.vocabulary_):,}"
    )
    print(
        "Accuracy:",
        f"{accuracy:.4f}"
    )
    print(
        "Precision macro:",
        f"{precision_macro:.4f}"
    )
    print(
        "Recall macro:",
        f"{recall_macro:.4f}"
    )
    print(
        "F1 macro:",
        f"{f1_macro:.4f}"
    )
    print(
        "Tiempo de vectorización:",
        f"{tiempo_vectorizacion:.2f} segundos"
    )

    # Liberar memoria antes del siguiente experimento
    del vectorizador
    del modelo_temporal
    del X_train_interno_tfidf
    del X_validacion_tfidf
    del predicciones_validacion

    gc.collect()


CONFIGURACIÓN 1/4
configuracion_base
--------------------------------------------------
Vocabulario: 50,000
Accuracy: 0.8523
Precision macro: 0.6696
Recall macro: 0.6167
F1 macro: 0.6352
Tiempo de vectorización: 259.21 segundos

CONFIGURACIÓN 2/4
solo_unigramas
--------------------------------------------------
Vocabulario: 48,877
Accuracy: 0.8290
Precision macro: 0.6483
Recall macro: 0.5109
F1 macro: 0.5368
Tiempo de vectorización: 35.36 segundos

CONFIGURACIÓN 3/4
vocabulario_ampliado
--------------------------------------------------
Vocabulario: 75,000
Accuracy: 0.8533
Precision macro: 0.6703
Recall macro: 0.6173
F1 macro: 0.6353
Tiempo de vectorización: 143.26 segundos

CONFIGURACIÓN 4/4
menor_frecuencia_minima
--------------------------------------------------
Vocabulario: 50,000
Accuracy: 0.8523
Precision macro: 0.6695
Recall macro: 0.6166
F1 macro: 0.6351
Tiempo de vectorización: 158.88 segundos


In [78]:
# guardar resultados 
resultados_ajuste_df = pd.DataFrame(
    resultados_ajuste
)

resultados_ajuste_df = (
    resultados_ajuste_df
    .sort_values(
        by="F1 macro",
        ascending=False
    )
    .reset_index(drop=True)
)

resultados_ajuste_df.to_csv(
    RUTA_SEGUNDA_ITERACION
    / "comparacion_configuraciones_tfidf.csv",
    index=False
)

resultados_ajuste_df.round(4)

,Configuración,max_features,min_df,max_df,ngram_range,sublinear_tf,Vocabulario real,Accuracy,Precision macro,Recall macro,F1 macro,Tiempo vectorización,Tiempo entrenamiento
0,vocabulario_ampliado,75000,5,0.95,"(1, 2)",True,75000,0.8533,0.6703,0.6173,0.6353,143.2555,5.1609
1,configuracion_base,50000,5,0.95,"(1, 2)",True,50000,0.8523,0.6696,0.6167,0.6352,259.2143,2.9029
2,menor_frecuencia_minima,50000,3,0.95,"(1, 2)",True,50000,0.8523,0.6695,0.6166,0.6351,158.8782,4.2548
3,solo_unigramas,50000,5,0.95,"(1, 1)",True,48877,0.8290,0.6483,0.5109,0.5368,35.3589,3.0020
